# AutoDL 实验运行 Notebook

在 AutoDL JupyterLab 里逐步运行所有实验。

> **使用前提**：已通过 scp / JupyterLab 上传 `data/billsum/` 和 `data/casehold/`

## 0. 环境准备

In [ ]:
import os
os.chdir('/root/MLP')
os.makedirs('logs', exist_ok=True)
!pwd
!ls


In [ ]:
%%bash
# AutoDL 已预装 PyTorch + CUDA，只需补装缺少的包
source /etc/network_turbo 2>/dev/null || true
pip install peft accelerate trl bitsandbytes wandb rouge-score bert-score scikit-learn sentencepiece huggingface_hub
echo "依赖安装完成"


In [ ]:
import torch, peft, trl
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')


In [ ]:
# 验证数据文件
import os
files = [
    'data/billsum/train_sft.jsonl',
    'data/billsum/val_sft.jsonl',
    'data/billsum/test_us_sft.jsonl',
    'data/billsum/test_ca_sft.jsonl',
    'data/casehold/train_mc.jsonl',
    'data/casehold/validation_mc.jsonl',
    'data/casehold/test_mc.jsonl',
]
for f in files:
    size = os.path.getsize(f) // 1024 if os.path.exists(f) else -1
    status = f'✓ {size} KB' if size >= 0 else '✗ 缺失！'
    print(f'{status}  {f}')

## 1. BillSum LoRA — Qwen

In [ ]:
%%bash
export WANDB_MODE=offline
export PYTHONUNBUFFERED=1
cd /root/MLP
python -u src/train/train.py --config configs/lora_billsum_qwen.yaml 2>&1 | tee logs/lora_billsum_qwen.log
echo "训练完成: $(date)"


In [ ]:
%%bash
export PYTHONUNBUFFERED=1
cd /root/MLP
python -u src/evaluate/inference.py --config configs/lora_billsum_qwen.yaml --split test_us
python -u src/evaluate/inference.py --config configs/lora_billsum_qwen.yaml --split test_ca
python -u src/evaluate/eval_billsum.py \
    --predictions outputs/lora_billsum_qwen/predictions_test_us.jsonl \
    --output outputs/lora_billsum_qwen/eval_test_us.json
python -u src/evaluate/eval_billsum.py \
    --predictions outputs/lora_billsum_qwen/predictions_test_ca.jsonl \
    --output outputs/lora_billsum_qwen/eval_test_ca.json
echo "评估完成"
cat outputs/lora_billsum_qwen/eval_test_us.json


## 2. BillSum LoRA — Llama

In [ ]:
import os, getpass
from huggingface_hub import login

token = os.getenv("HF_TOKEN") or getpass.getpass("请输入 HuggingFace token（不会回显）: ")
login(token=token)
print("HuggingFace 登录成功")


In [ ]:
%%bash
export WANDB_MODE=offline
export PYTHONUNBUFFERED=1
cd /root/MLP
python -u src/train/train.py --config configs/lora_billsum_llama.yaml 2>&1 | tee logs/lora_billsum_llama.log
python -u src/evaluate/inference.py --config configs/lora_billsum_llama.yaml --split test_us
python -u src/evaluate/inference.py --config configs/lora_billsum_llama.yaml --split test_ca
python -u src/evaluate/eval_billsum.py \
    --predictions outputs/lora_billsum_llama/predictions_test_us.jsonl \
    --output outputs/lora_billsum_llama/eval_test_us.json
python -u src/evaluate/eval_billsum.py \
    --predictions outputs/lora_billsum_llama/predictions_test_ca.jsonl \
    --output outputs/lora_billsum_llama/eval_test_ca.json
echo "完成: $(date)"
cat outputs/lora_billsum_llama/eval_test_us.json


## 3. BillSum Full FT — Qwen

In [ ]:
%%bash
export WANDB_MODE=offline
export PYTHONUNBUFFERED=1
cd /root/MLP
python -u src/train/train.py --config configs/full_billsum_qwen.yaml 2>&1 | tee logs/full_billsum_qwen.log
python -u src/evaluate/inference.py --config configs/full_billsum_qwen.yaml --split test_us
python -u src/evaluate/inference.py --config configs/full_billsum_qwen.yaml --split test_ca
python -u src/evaluate/eval_billsum.py \
    --predictions outputs/full_billsum_qwen/predictions_test_us.jsonl \
    --output outputs/full_billsum_qwen/eval_test_us.json
python -u src/evaluate/eval_billsum.py \
    --predictions outputs/full_billsum_qwen/predictions_test_ca.jsonl \
    --output outputs/full_billsum_qwen/eval_test_ca.json
echo "完成: $(date)"
cat outputs/full_billsum_qwen/eval_test_us.json


## 4. BillSum Full FT — Llama

In [ ]:
%%bash
export WANDB_MODE=offline
export PYTHONUNBUFFERED=1
cd /root/MLP
python -u src/train/train.py --config configs/full_billsum_llama.yaml 2>&1 | tee logs/full_billsum_llama.log
python -u src/evaluate/inference.py --config configs/full_billsum_llama.yaml --split test_us
python -u src/evaluate/inference.py --config configs/full_billsum_llama.yaml --split test_ca
python -u src/evaluate/eval_billsum.py \
    --predictions outputs/full_billsum_llama/predictions_test_us.jsonl \
    --output outputs/full_billsum_llama/eval_test_us.json
python -u src/evaluate/eval_billsum.py \
    --predictions outputs/full_billsum_llama/predictions_test_ca.jsonl \
    --output outputs/full_billsum_llama/eval_test_ca.json
echo "完成: $(date)"
cat outputs/full_billsum_llama/eval_test_us.json


## 5. CaseHOLD LoRA — Qwen

In [ ]:
%%bash
export WANDB_MODE=offline
export PYTHONUNBUFFERED=1
cd /root/MLP
python -u src/train/train.py --config configs/lora_casehold_qwen.yaml 2>&1 | tee logs/lora_casehold_qwen.log
python -u src/evaluate/inference.py --config configs/lora_casehold_qwen.yaml --split test
python -u src/evaluate/eval_casehold.py \
    --predictions outputs/lora_casehold_qwen/predictions_test.jsonl \
    --output outputs/lora_casehold_qwen/eval_test.json
echo "完成: $(date)"
cat outputs/lora_casehold_qwen/eval_test.json


## 6. CaseHOLD LoRA — Llama

In [ ]:
%%bash
export WANDB_MODE=offline
export PYTHONUNBUFFERED=1
cd /root/MLP
python -u src/train/train.py --config configs/lora_casehold_llama.yaml 2>&1 | tee logs/lora_casehold_llama.log
python -u src/evaluate/inference.py --config configs/lora_casehold_llama.yaml --split test
python -u src/evaluate/eval_casehold.py \
    --predictions outputs/lora_casehold_llama/predictions_test.jsonl \
    --output outputs/lora_casehold_llama/eval_test.json
echo "完成: $(date)"
cat outputs/lora_casehold_llama/eval_test.json


## 7. CaseHOLD QLoRA — Qwen

In [ ]:
%%bash
export WANDB_MODE=offline
export PYTHONUNBUFFERED=1
cd /root/MLP
python -u src/train/train_casehold_lora.py --config configs/qlora_casehold_qwen.yaml 2>&1 | tee logs/qlora_casehold_qwen.log
python -u src/evaluate/inference.py --config configs/qlora_casehold_qwen.yaml --split test
python -u src/evaluate/eval_casehold.py \
    --predictions outputs/qlora_casehold_qwen/predictions_test.jsonl \
    --output outputs/qlora_casehold_qwen/eval_test.json
echo "完成: $(date)"
cat outputs/qlora_casehold_qwen/eval_test.json


## 8. CaseHOLD QLoRA — Llama

In [ ]:
%%bash
export WANDB_MODE=offline
export PYTHONUNBUFFERED=1
cd /root/MLP
python -u src/train/train_casehold_lora.py --config configs/qlora_casehold_llama.yaml 2>&1 | tee logs/qlora_casehold_llama.log
python -u src/evaluate/inference.py --config configs/qlora_casehold_llama.yaml --split test
python -u src/evaluate/eval_casehold.py \
    --predictions outputs/qlora_casehold_llama/predictions_test.jsonl \
    --output outputs/qlora_casehold_llama/eval_test.json
echo "完成: $(date)"
cat outputs/qlora_casehold_llama/eval_test.json


## 汇总所有结果

In [ ]:
import json, os

results = [
    ('lora_billsum_qwen',   'outputs/lora_billsum_qwen/eval_test_us.json'),
    ('lora_billsum_llama',  'outputs/lora_billsum_llama/eval_test_us.json'),
    ('full_billsum_qwen',   'outputs/full_billsum_qwen/eval_test_us.json'),
    ('full_billsum_llama',  'outputs/full_billsum_llama/eval_test_us.json'),
    ('lora_casehold_qwen',  'outputs/lora_casehold_qwen/eval_test.json'),
    ('lora_casehold_llama', 'outputs/lora_casehold_llama/eval_test.json'),
    ('qlora_casehold_qwen', 'outputs/qlora_casehold_qwen/eval_test.json'),
    ('qlora_casehold_llama','outputs/qlora_casehold_llama/eval_test.json'),
]

print(f'{"实验":<25} {"指标":<15} {"值":>8}')
print('-' * 52)
for name, path in results:
    if not os.path.exists(path):
        print(f'{name:<25} {"未完成":<15}')
        continue
    with open(path) as f:
        d = json.load(f)
    # BillSum 看 rouge2，CaseHOLD 看 accuracy
    if 'rouge2' in d:
        print(f'{name:<25} {"rouge2":<15} {d["rouge2"]:>8.4f}')
    elif 'accuracy' in d:
        print(f'{name:<25} {"accuracy":<15} {d["accuracy"]:>8.4f}')